# Python Mini Rules

Ein einfacher Regelinterpreter ohne Konfliktlösung und Backtracking für aussagenlogische Anwendungen.
Anmerkung: dies ist eher eine Vorstufe zur KI, da Regelinterpreter in Produktionssystemen ansonsten deutlich anspruchsvoller sind. Das betrifft die Konfliktlösung, Backtracking und effizienzsteigernde Maßnahmen. Kern von solchen Systemen ist dann auch der RETE Algorithmus. 


In [1]:
# Leerer Wissensspeicher - keine Fakten vorhanden

memory = set()

# Leere Regelmenge - keine Regeln vorhanden

rules  = list()

### Einfache Regelrepräsentation

Aussagenlogische Variablen als Menge von Voraussetzungen und Menge von Konsequenzen. Die zwei wesentlichen Punkte sind die Anwendungsprüfung (match) und die Anwendung der Regel (apply). Bei einem anderen Einsatz der Regeln sind diese anzupassen.

In [2]:
class rule:
    name = ""
    head = set()
    tail = set()
    
    def __init__(self,n,X,Y):
        self.name = n
        self.head = X
        self.tail = Y
        
    def match(self,s):
        if self.head.issubset(s):
            return True
        
    def apply(self,memory):
        return memory.union(self.tail)
        
    def print(self):
        out = self.name + " " + str(self.head) + "->" + str(self.tail)
        print(out)

### Beispielregeln 
Die Regeln werden benannt und sie bestehen jeweils aus Voraussetzungsmengen von Aussagevariablen (head) und Schlussfolgerungen aus diesen (tail). 

In [3]:
rules.append(rule("R1",set('ab'),set('cf')))
rules.append(rule("R2",set('bd'),set('ac')))
rules.append(rule("R3",set('c'),set('de')))
rules.append(rule("R4",set('ac'),set('b')))

Nun werden Aussagevariablen gesetzt (bzw. "Beobachtungen gemacht").

In [4]:
memory.add('a')
memory.add('c')

Die Regelanwendung wird geprüft und nacheinander alle Regeln angewandt, deren Bedingung erfüllt wird. Dadurch entstehen (Forward Chaining) neue Daten, die ihrerseits von Regeln verwendet werden können. Sobald keine neuen Daten mehr durch die Regeln erzeugt werden können terminiert das Regelsystem.


In [5]:
n = len(memory)
n_old = n-1

while (n > n_old):
    print("Wissensspeicher:")
    print(memory)

    for r in rules:
        if r.match(memory):
            # Regel anwenden
            out = " -> Regel " + r.name + " feuert. Hinzufügen: " + str(r.tail)
            print(out)
            memory = r.apply(memory)
        
    print("Wissensspeicher (neu):")
    print(memory)
    n_old = n
    n = len(memory)

Wissensspeicher:
{'a', 'c'}
 -> Regel R3 feuert. Hinzufügen: {'d', 'e'}
 -> Regel R4 feuert. Hinzufügen: {'b'}
Wissensspeicher (neu):
{'c', 'd', 'a', 'b', 'e'}
Wissensspeicher:
{'c', 'd', 'a', 'b', 'e'}
 -> Regel R1 feuert. Hinzufügen: {'c', 'f'}
 -> Regel R2 feuert. Hinzufügen: {'a', 'c'}
 -> Regel R3 feuert. Hinzufügen: {'d', 'e'}
 -> Regel R4 feuert. Hinzufügen: {'b'}
Wissensspeicher (neu):
{'c', 'd', 'a', 'b', 'f', 'e'}
Wissensspeicher:
{'c', 'd', 'a', 'b', 'f', 'e'}
 -> Regel R1 feuert. Hinzufügen: {'c', 'f'}
 -> Regel R2 feuert. Hinzufügen: {'a', 'c'}
 -> Regel R3 feuert. Hinzufügen: {'d', 'e'}
 -> Regel R4 feuert. Hinzufügen: {'b'}
Wissensspeicher (neu):
{'c', 'd', 'a', 'b', 'f', 'e'}


## Anwendungsbeispiel: Die Attributhülle 

Beim relationalen Datenbankentwurf kann man mit der Attributhüllenbestimmung beispielsweise bestimmen, ob eine Attributkombination ein möglicher Schlüsselkandidat ist. 

Gegeben sei eine Relation R (a,b,c,d,e,f) sowie die funktionalen Abhängigkeiten FD = { d->ba,a->ed,b->af,bc->e }

Es wird vermutet, dass bc ein Schlüssel ist. Die Attributhülle AH(FD,{b,c}) wird nun bestimmt. Wenn diese alle Attribute der Relation enthält ist bc ein Schlüssel (sofern nicht b oder c alleine Schlüssel sind -> Minimalitätsprinzip).

In [6]:
def ah(fd,attr):
    n = len(fd)
    n_old = n-1

    while (n > n_old):
        for r in fd:
            if r.match(attr):
                attr = r.apply(attr)
        n_old = n
        n = len(attr)
    return attr

FD setzen

In [7]:
fd = list()
fd.append(rule("R1",set('d'),set('ba')))
fd.append(rule("R2",set('a'),set('ed')))
fd.append(rule("R3",set('b'),set('af')))
fd.append(rule("R4",set('bc'),set('e')))

In [8]:
attr = set('bc')

In [9]:
ah(fd,attr)

{'a', 'b', 'c', 'd', 'e', 'f'}